# Part 3: VALIDATION, PREDICTION, EVALUATION AND INTERPRETATION


Role: PPT Lead

Depends on: X_train, X_test, y_train, y_test and the fitted model 

Hands over: Cross-validation scores, test predictions, evaluation metrics, coefficient table and interpretation.



## 1. Introduction

This phase focuses on validating and evaluating the Linear Regression model developed in the previous stages of the project.

The main objective is to assess the model's predictive performance, examine its ability to generalize to unseen data, and interpret the factors influencing life expectancy.

The following tasks are performed:

- Model validation using 5-Fold Cross-Validation.
- Prediction of life expectancy on the test dataset.
- Evaluation using appropriate regression metrics.
- Comparison of actual and predicted values.
- Analysis and interpretation of model coefficients.


In [ ]:

# 2. Importing Required Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

print("All required libraries imported successfully.")


In [ ]:
import joblib
from pathlib import Path

data_path = Path("part3_outputs/part3_data.pkl")

part3_data = joblib.load(data_path)

model = part3_data["model"]
X_train = part3_data["X_train"]
X_test = part3_data["X_test"]
y_train = part3_data["y_train"]
y_test = part3_data["y_test"]

print("Model and datasets loaded successfully!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 2. Cross-validation

Five-fold cross-validation is performed on the training data. Each fold uses four parts for fitting and one part for validation. The test set remains untouched for the final evaluation.

In [ ]:

def cross_validate_model(model, X_train, y_train):
    """Perform 5-fold cross-validation and return R² scores."""

    kfold = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=kfold,
        scoring="r2"
    )

    print("R² Scores for each fold:")
    for i, score in enumerate(cv_scores, start=1):
        print(f"Fold {i}: {score:.4f}")

    print(f"\nMean R² Score: {cv_scores.mean():.4f}")
    print(f"Standard Deviation: {cv_scores.std():.4f}")

    return cv_scores


# Execute cross-validation
cv_scores = cross_validate_model(model, X_train, y_train)


## 3. Test-set prediction and evaluation

The fitted pipeline predicts the held-out test set. MAE, MSE, RMSE and R2 are calculated from these predictions. The comparison table displays the first ten test observations.

In [ ]:

def evaluate_model(model, X_test, y_test):
    """
    Generate test predictions and calculate regression metrics.
    """

    # Generate predictions
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Display results
    print("Model Evaluation Results")
    print("-------------------------")
    print(f"Number of predictions: {len(y_pred)}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}")

    return y_pred, {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }


# Evaluate the trained model
y_pred, evaluation_results = evaluate_model(
    model, X_test, y_test
)


## 4. Actual vs Predicted analysis

Points close to the diagonal line represent predictions close to the actual values. This plot is a visual check of test-set performance.

In [ ]:

# Create a comparison of actual and predicted values

comparison_df = pd.DataFrame({
    "Actual Life Expectancy": y_test.values,
    "Predicted Life Expectancy": y_pred
})

comparison_df["Prediction Error"] = (
    comparison_df["Actual Life Expectancy"]
    - comparison_df["Predicted Life Expectancy"]
)

# Display the first 10 predictions
comparison_df.head(10)


In [ ]:

# Actual vs. Predicted Visualization

plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, alpha=0.6)

# Reference line for perfect predictions
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    linewidth=2,
    label="Perfect Prediction"
)

plt.xlabel("Actual Life Expectancy (Years)")
plt.ylabel("Predicted Life Expectancy (Years)")
plt.title("Actual vs. Predicted Life Expectancy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 5. Model Coefficients Analysis

The coefficients are extracted after preprocessing. Because the pipeline standardizes the transformed features, coefficient magnitudes describe the model's change in predicted years for a one-standard-deviation change in a transformed feature, holding the other features fixed. These are associations in the fitted model, not proof of causation.

In [ ]:

def get_coefficients(model):
    """
    Extract feature names and their corresponding
    Linear Regression coefficients.
    """

    feature_names = (
        model.named_steps["preprocessor"].get_feature_names_out()
    )

    coefficients = model.named_steps["regressor"].coef_

    coefficient_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients
    })

    coefficient_df["Absolute Coefficient"] = (
        coefficient_df["Coefficient"].abs()
    )

    coefficient_df = coefficient_df.sort_values(
        by="Absolute Coefficient",
        ascending=False
    ).reset_index(drop=True)

    return coefficient_df


# Extract and display coefficients
coefficient_df = get_coefficients(model)

print("Feature Coefficient Analysis")
display(coefficient_df)


In [ ]:
# Top 5 Positive Factors
positive_factors = coefficient_df[
    coefficient_df["Coefficient"] > 0
].nlargest(5, "Coefficient")[["Feature", "Coefficient"]]

# Top 5 Negative Factors
negative_factors = coefficient_df[
    coefficient_df["Coefficient"] < 0
].nsmallest(5, "Coefficient")[["Feature", "Coefficient"]]

print("Top 5 Positive Factors")
display(positive_factors.reset_index(drop=True))

print("Top 5 Negative Factors")
display(negative_factors.reset_index(drop=True))


## 6. Model Interpretation And Key Findings

Based on the coefficient analysis, the following features show the strongest relationships with predicted life expectancy:

**Top Positive Factors:**
- Schooling (+2.1994): Shows the strongest positive association with predicted life expectancy.
- Income composition of resources (+1.2182): Indicates a positive association.
- Diphtheria (+1.1396): Also shows a positive association.

**Top Negative Factors:**
- Adult Mortality (-2.5976): Shows the strongest negative association with predicted life expectancy.
- HIV/AIDS (-2.5387): Shows another strong negative association.
- Status_Developing (-0.5490): Has a negative coefficient in the fitted model.

These coefficients represent associations within the Linear Regression model and should not be interpreted as proof of causation.

### Interpretation of Prediction Error (RMSE)
The RMSE of 4.0583 indicates that the model's prediction errors have a typical magnitude of approximately 4.06 years of life expectancy.


In [ ]:

# Visualize the top 10 influential features

top_features = coefficient_df.head(10).sort_values(
    by="Coefficient"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Coefficient"]
)

plt.axvline(x=0, color="black", linewidth=0.8)
plt.xlabel("Regression Coefficient")
plt.ylabel("Features")
plt.title("Top 10 Features Influencing Life Expectancy")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()



## 7. Conclusion

The Linear Regression model achieved an R² score of 0.8096 on the test dataset, demonstrating its ability to explain a substantial proportion of the variation in life expectancy.

The 5-Fold Cross-Validation results showed a mean R² score of 0.8066 with a standard deviation of 0.0121, indicating consistent model performance across the validation folds.

The model recorded an MAE of 3.0163 years and an RMSE of 4.0583 years.

The coefficient analysis highlighted Adult Mortality and HIV/AIDS as the strongest negative factors, while Schooling showed the strongest positive association with predicted life expectancy.

Overall, the model provides reasonably good predictions, although further improvements may be possible through more advanced regression techniques and additional features.
